# 02 · Dil Tespiti ve Veri Bölme

**Proje:** Akıllı Gıda Etiketi Analizi
**Bu not defterinin amacı:** Ürün metinlerinin dilini bulmak ve veriyi eğitim / doğrulama / test kümelerine **sızıntısız** biçimde ayırmak.

| | |
|---|---|
| **Girdi** | `dataset.parquet` (01 not defterinin çıktısı) |
| **Çıktı 1** | `dataset_dil.parquet` — dil sütunları eklenmiş veri kümesi |
| **Çıktı 2** | `split.csv` — her ürünün hangi kümede olduğu (`code`, `kume`, `kume_rastgele`) |
| **Çıktı 3** | `reports/` altında dil ve bölme tabloları |

**Akış:**
1. Kurulum
2. Dil tespiti
3. Marka anahtarı
4. Marka bazlı bölme (ana bölme)
5. Rastgele bölme (karşılaştırma için)
6. Kontroller: sızıntı var mı?
7. Kaydetme

## 1. Kurulum

**Dil tespiti için `lingua` kütüphanesi** kullanılıyor. Ödev metninde fastText önerilmişti; ancak `fasttext` paketi yeni Python sürümlerinde derleme sorunları çıkarabiliyor. `lingua` hazır kurulur, kısa metinlerde de doğruluğu yüksektir ve Türkçeyi destekler. Bu tercihi raporda belirt.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q lingua-language-detector==2.2.0 duckdb==1.5.6

import os
import numpy as np
import pandas as pd

BASE    = "/content/drive/MyDrive/off"
DS      = f"{BASE}/dataset.parquet"
DS_DIL  = f"{BASE}/dataset_dil.parquet"
SPLIT   = f"{BASE}/split.csv"
REPORTS = f"{BASE}/reports"
os.makedirs(REPORTS, exist_ok=True)

SEED = 42

df = pd.read_parquet(DS)
print(f"{len(df):,} ürün")
df.grup.value_counts()

## 2. Dil tespiti

**Neden?**
- `TR_barkod` ürünleri çoğunlukla Fransa, Almanya ve Irak'ta kayıtlı; etiketleri Fransızca, Almanca veya Arapça olabilir.
- Avrupa ürünleri farklı dillerde.
- Türkçe modellerin (BERTurk) hangi verilerle eğitileceğine ve sonuçların dil bazında nasıl raporlanacağına bu bilgiyle karar verilecek.

**Nasıl?**
- Dedektör yalnızca veride beklenen dillerle sınırlanır. Bu, kısa metinlerde doğruluğu artırır.
- Her metin için **en olası dil** ve **güven skoru** (0–1) kaydedilir. Güven 0,5'in altındaysa dil `belirsiz` sayılır; örneğin çok dilli etiketlerde veya çok kısa metinlerde.
- Hem içindekiler metninin (`ic_dil`) hem ürün adının (`ad_dil`) dili bulunur.

`detect_multiple_languages_in_parallel_of` hazır bir toplu işlem fonksiyonudur; bütün işlemci çekirdeklerini kullanır.

In [ ]:
from lingua import Language, LanguageDetectorBuilder

DILLER = [Language.TURKISH, Language.ENGLISH, Language.FRENCH, Language.GERMAN,
          Language.SPANISH, Language.ITALIAN, Language.DUTCH, Language.ARABIC,
          Language.PORTUGUESE, Language.POLISH, Language.ROMANIAN, Language.BULGARIAN,
          Language.RUSSIAN, Language.GREEK]
detector = LanguageDetectorBuilder.from_languages(*DILLER).with_preloaded_language_models().build()

ESIK = 0.5

def dil_bul(metinler):
    # Her metin için (dil_kodu, güven) döndürür; boş metinler için (None, None)
    sonuc = [(None, None)] * len(metinler)
    dolu = [(i, str(m)[:500]) for i, m in enumerate(metinler) if isinstance(m, str) and m.strip()]
    if not dolu:
        return sonuc
    guvenler = detector.compute_language_confidence_values_in_parallel([m for _, m in dolu])
    for (i, _), degerler in zip(dolu, guvenler):
        en_iyi = degerler[0]
        kod = en_iyi.language.iso_code_639_1.name.lower()
        sonuc[i] = (kod if en_iyi.value >= ESIK else "belirsiz", round(en_iyi.value, 3))
    return sonuc

df[["ic_dil", "ic_dil_guven"]] = pd.DataFrame(dil_bul(df.ingredients_text.tolist()), index=df.index)
df[["ad_dil", "ad_dil_guven"]] = pd.DataFrame(dil_bul(df.product_name.tolist()), index=df.index)
print("Bitti.")

### Dil dağılımı (içindekiler metni)

**Beklenen:** `TR_ulke` grubunda Türkçe baskın; `TR_barkod` grubunda Türkçe, Fransızca, Almanca, Arapça karışık; `AB` grubunda Fransızca baskın.

In [ ]:
dil_tablosu = (df[df.ic_dil.notna()]
               .groupby(["grup", "ic_dil"]).size()
               .unstack(fill_value=0))
dil_tablosu = dil_tablosu[dil_tablosu.sum().sort_values(ascending=False).index]
dil_tablosu.to_csv(f"{REPORTS}/dil_dagilimi.csv")
display(dil_tablosu)
display((dil_tablosu.div(dil_tablosu.sum(axis=1), axis=0) * 100).round(1))

### Göz kontrolü

Otomatik dil tespitine körü körüne güvenmemek için her dilden birkaç örnek. Yanlış tespit edilen bir şey varsa raporda belirt.

In [ ]:
pd.set_option("display.max_colwidth", 90)
(df[df.ic_dil.notna()]
   .sample(frac=1, random_state=SEED)      # karıştır
   .groupby("ic_dil").head(3)              # her dilden ilk 3
   .sort_values("ic_dil")
   [["grup", "ic_dil", "ic_dil_guven", "ingredients_text"]])

## 3. Marka anahtarı

**Neden marka bazlı bölme?**
Aynı markanın ürünleri birbirine çok benzer: aynı ambalaj tasarımı, aynı tarif kalıpları, benzer ürün adları ("Ülker Çikolatalı Gofret", "Ülker Fındıklı Gofret"). Bir markanın ürünleri hem eğitim hem test kümesine düşerse model **markayı ezberleyerek** yüksek skor alır. Bu, gerçek dünyadaki başarımı olduğundan iyi gösterir: bir tür **sızıntı**.

Marka bazlı bölmede bir markanın bütün ürünleri **yalnızca tek bir kümede** bulunur. Model test kümesinde hiç görmediği markalarla sınanır.

**Marka anahtarı nasıl belirleniyor?**
- `brands_tags` sütunundaki **ilk** marka alınır (örneğin `ulker,godiva` → `ulker`).
- Markası boş olan ürünler kendi barkodlarıyla ayrı bir grup sayılır. Böylece markasız ürünlerin hepsi tek dev bir gruba toplanmaz.
- Etiketlerin başındaki dil öneki (`xx:`) temizlenir.

> **Sınırlama (rapor):** Aynı marka farklı yazımlarla geçebiliyor (örneğin `marks-spencer`, `m-s`, `marks-spencers`). Bu durumda aynı markanın ürünleri farklı kümelere düşebilir; yani marka sızıntısı tamamen sıfırlanamaz. Aşağıdaki listede bu tür yazım farklarına göz at.

In [ ]:
ilk_marka = (df.brands_tags.fillna("").str.split(",").str[0].str.strip()
             .str.replace(r"^[a-z]{2}:", "", regex=True))       # "xx:ulker" -> "ulker"
df["marka_anahtari"] = np.where(ilk_marka != "", ilk_marka, "markasiz_" + df.code)

print(f"Farklı marka sayısı: {df.marka_anahtari.nunique():,}")
print(f"Markası boş ürün: {(ilk_marka == '').sum():,}\n")
print("En çok ürünü olan 25 marka:")
df.marka_anahtari.value_counts().head(25)

## 4. Marka bazlı bölme (ana bölme)

**Araç:** scikit-learn'ün `StratifiedGroupKFold` sınıfı. İki şeyi aynı anda yapar:
- **Group (grup):** Aynı markanın ürünleri aynı parçada kalır.
- **Stratified (tabakalı):** Her parçada sınıf oranları mümkün olduğunca korunur.

**Tabaka etiketi:** `grup + Nutri-Score` (örneğin `TR_ulke_d`, `AB_a`). Böylece hem Türkiye/Avrupa oranı hem Nutri-Score dağılımı üç kümede benzer olur. Nutri-Score'u olmayan ürünler `yok` sınıfında.

**%70 / %15 / %15 nasıl elde ediliyor?** Veri 20 parçaya bölünür; 14 parça eğitim (%70), 3 parça doğrulama (%15), 3 parça test (%15) olur.

> Büyük markalar tek parçaya sığmak zorunda olduğu için oranlar tam %70/15/15 çıkmayabilir; birkaç puanlık sapma normaldir. Gerçek oranlar kontrol adımında raporlanır.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

tabaka = df.grup + "_" + df.nutriscore_grade.where(df.ns_var, "yok").fillna("yok")

sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
parca = np.empty(len(df), dtype=int)
for i, (_, test_idx) in enumerate(sgkf.split(df, tabaka, groups=df.marka_anahtari)):
    parca[test_idx] = i

df["kume"] = np.select([parca < 14, parca < 17], ["train", "val"], default="test")
print((df.kume.value_counts(normalize=True) * 100).round(1))

## 5. Rastgele bölme (karşılaştırma için)

Ödevde istenen ikinci strateji: markaya bakmadan, yalnızca tabakalı rastgele bölme. Ana deneylerde **kullanılmayacak**. Aynı modeli iki bölmeyle eğitip test skorları karşılaştırılacak (Tablo 1, satır 13).

**Beklenen:** Rastgele bölmede skor daha **yüksek** çıkar, çünkü model test kümesindeki markaları eğitimde görmüştür. Aradaki fark, marka sızıntısının büyüklüğünü gösterir.

In [ ]:
from sklearn.model_selection import train_test_split

# Tabaka etiketi 4. bölümdekiyle aynı; hücre tek başına da çalışsın diye burada yeniden tanımlanıyor
tabaka = df.grup + "_" + df.nutriscore_grade.where(df.ns_var, "yok").fillna("yok")

def nadirleri_birlestir(t, en_az):
    # Çok az örneği olan tabakalar tabakalı bölmeyi bozar; onları en kalabalık tabakaya katarız
    say = t.value_counts()
    return t.where(t.map(say) >= en_az, say.index[0])

tabaka_r = nadirleri_birlestir(tabaka, 10)
idx_train, idx_kalan = train_test_split(df.index, test_size=0.30, stratify=tabaka_r, random_state=SEED)

tabaka_k = nadirleri_birlestir(tabaka_r[idx_kalan], 2)
idx_val, idx_test = train_test_split(idx_kalan, test_size=0.50, stratify=tabaka_k, random_state=SEED)

df["kume_rastgele"] = "train"
df.loc[idx_val, "kume_rastgele"] = "val"
df.loc[idx_test, "kume_rastgele"] = "test"
print((df.kume_rastgele.value_counts(normalize=True) * 100).round(1))

## 6. Kontroller: sızıntı var mı?

1. **Marka sızıntısı:** Ana bölmede hiçbir marka birden fazla kümede olmamalı. Rastgele bölmede ise olacak; kaç tane olduğu raporlanır.
2. **Aynı ürün adı:** Aynı ürün adı (küçük harf, boşluklar temizlenmiş) farklı kümelerde geçiyor mu? Ödevin istediği kontrol bu. Örneğin aynı ürünün farklı gramajları farklı barkodlarla kayıtlı olabilir.
3. **Dağılımlar:** Her kümede grup ve Nutri-Score oranları benzer mi?

In [ ]:
def kume_sayisi(sutun, anahtar):
    return df.groupby(anahtar)[sutun].nunique()

marka_ana = (kume_sayisi("kume", "marka_anahtari") > 1).sum()
marka_rst = (kume_sayisi("kume_rastgele", "marka_anahtari") > 1).sum()

ad_norm = df.product_name.fillna("").str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
df["_ad"] = ad_norm
dolu_ad = df[df._ad != ""]
ad_ana = (dolu_ad.groupby("_ad").kume.nunique() > 1).sum()
ad_rst = (dolu_ad.groupby("_ad").kume_rastgele.nunique() > 1).sum()

kontrol = pd.DataFrame({
    "marka_bazli": [marka_ana, ad_ana],
    "rastgele":    [marka_rst, ad_rst],
}, index=["Birden fazla kümede geçen marka", "Birden fazla kümede geçen ürün adı"])
kontrol.to_csv(f"{REPORTS}/sizinti_kontrolu.csv")
display(kontrol)

assert marka_ana == 0, "Marka bazlı bölmede marka sızıntısı var!"
print("✅ Marka bazlı bölmede marka sızıntısı yok.")

Aynı ürün adının marka bazlı bölmede de birden fazla kümede geçmesi mümkündür: farklı markaların aynı genel adı kullanması ("Süzme Yoğurt", "Ayran"). Bu durum sızıntı sayılmaz, ama örneklere göz atmakta fayda var:

In [ ]:
coklu = dolu_ad.groupby("_ad").filter(lambda g: g.kume.nunique() > 1)
coklu.sort_values("_ad")[["_ad", "marka_anahtari", "kume", "grup"]].head(20)

### Kümelerin dağılımı

Üç kümede grup ve Nutri-Score oranlarının benzer olması beklenir. Büyük farklar varsa, tabakalamanın büyük markalar yüzünden bozulduğu anlamına gelir.

In [ ]:
grup_dagilim = pd.crosstab(df.grup, df.kume)[["train", "val", "test"]]
grup_dagilim.to_csv(f"{REPORTS}/bolme_grup_dagilimi.csv")
display(grup_dagilim)

ns = df[df.ns_var]
ns_dagilim = (pd.crosstab(ns.kume, ns.nutriscore_grade, normalize="index") * 100).round(1).loc[["train", "val", "test"]]
ns_dagilim.to_csv(f"{REPORTS}/bolme_nutriscore_dagilimi.csv")
display(ns_dagilim)

### Türkiye test kümesi ne kadar büyük?

Türkiye sonuçlarının ne kadar güvenilir olacağını görmek için Görev A'ya uygun Türkiye ürünlerinin kümelere dağılımı. Bu sayılar küçükse sonuçlar raporda bootstrap güven aralığıyla verilecek.

In [ ]:
gorev_a = df.ns_var & df.ad & df.on_gorsel & df.metin
pd.crosstab(df[gorev_a].grup, df[gorev_a].kume)[["train", "val", "test"]]

## 7. Kaydetme

- `split.csv`: Ödevin istediği bölme dosyası. **Bütün modeller bu dosyayı kullanacak.** GitHub deposuna da (`data/split.csv`) kopyalanmalı.
- `dataset_dil.parquet`: Dil sütunları ve marka anahtarı eklenmiş veri kümesi. Sonraki not defterleri bu dosyadan başlayacak.

In [ ]:
df = df.drop(columns=["_ad"])
df[["code", "kume", "kume_rastgele"]].to_csv(SPLIT, index=False)
df.to_parquet(DS_DIL, index=False)

print("Kaydedildi:")
for yol in [SPLIT, DS_DIL]:
    print(f"  {yol}  ({os.path.getsize(yol)/1e6:.1f} MB)")

## Özet ve sonraki adım

**Bu not defterinde:**
- İçindekiler metinlerinin ve ürün adlarının dili bulundu.
- Veri marka bazlı olarak %70 / %15 / %15 bölündü; karşılaştırma için rastgele bölme de oluşturuldu.
- Marka sızıntısı olmadığı doğrulandı; dağılımlar raporlandı.

**Sonraki not defteri — `03_eda.ipynb`:** Ödevin istediği keşifsel analiz grafikleri (sınıf dağılımı, metin uzunlukları, örnek görsel ızgarası).